In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260108_bgs5_filtered_snm3_3T_L3.h5ad")

#3T --> bgs5_filtered hat was made just now
bgs5

In [ ]:
#resegmented

bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

bgs4

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Use your AnnData object
adata = bgs4  # or bgs4_lge_mge if that's the one you're using

# Get cluster annotation
clusters = adata.obs['adjusted_L3_transferred']
cluster_str = clusters.astype(str)

# Exclude 'uncertain' cells entirely
mask_known = cluster_str != 'uncertain'
cluster_str_filtered = cluster_str[mask_known]

# Get unique known clusters and sort nicely (numbers first, then others)
numeric = pd.to_numeric(clusters[mask_known], errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))
non_numeric_unique = np.sort(clusters[mask_known & ~is_numeric].unique())

all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Use pre-defined colors if available (recommended!)
if ('adjusted_L3_transferred' in adata.obs and
    adata.obs['adjusted_L3_transferred'].dtype.name == 'category' and
    'adjusted_L3_transferred_colors' in adata.uns):

    categories = adata.obs['adjusted_L3_transferred'].cat.categories
    known_categories = [cat for cat in categories if str(cat) != 'uncertain']
    known_categories_str = [str(cat) for cat in known_categories]
    color_list = adata.uns['adjusted_L3_transferred_colors']
    # Match colors to known categories
    cat_to_color = dict(zip([str(cat) for cat in categories], color_list))
    cluster_colors = {clust: cat_to_color.get(clust, 'gray') for clust in all_unique}
    print("Using pre-defined colors from adata.uns")
else:
    # Fallback: generate palette
    palette = sns.color_palette('tab20', len(all_unique))
    cluster_colors = dict(zip(all_unique, palette))
    print("Using generated color palette")

# Coordinates (filtered to exclude uncertain)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial'][mask_known]
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'][mask_known]
    y_spatial = adata.obs['CenterY_global_px'][mask_known]

umap_coords = adata.obsm['X_umap'][mask_known]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=3, edgecolor=None, ax=axes[0], legend=False)

axes[0].set_title('Adjusted L3 Clusters - Spatial (uncertain excluded)')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[1], legend=False)

axes[1].set_title('Adjusted L3 Clusters - UMAP (uncertain excluded)')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared external legend
handles = [Line2D([0], [0], marker='o', color='w',
                  markerfacecolor=cluster_colors[clust],
                  markersize=10) for clust in all_unique]

fig.legend(handles=handles, labels=all_unique,
           title='Adjusted L3 Clusters',
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

#2T-->bgs4_reseg ('harmony_failed/uncertain' cluster hidden)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs4
categ = 'adjusted_L3_transferred'

# Get unique clusters
clusters = sorted(adata.obs[categ].unique())
n_clusters = len(clusters)

# Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten() if n_clusters > 1 else [axes]

# Get spatial coordinates
spatial_coords = adata.obsm['spatial']

# Get colors for clusters
if f'{categ}_colors' in adata.uns:
    cluster_colors = dict(zip(adata.obs[categ].cat.categories, adata.uns[f'{categ}_colors']))
else:
    from matplotlib import cm
    cluster_colors = {c: cm.tab20(i % 20) for i, c in enumerate(clusters)}

grey_color = '#D3D3D3'
spot_size = 0.5

# Plot each cluster
for idx, cluster in enumerate(clusters):
    ax = axes[idx]

    # Create mask for this cluster
    mask = adata.obs[categ] == cluster

    # Plot grey background (all other cells)
    ax.scatter(
        spatial_coords[~mask, 0],
        spatial_coords[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.5
    )

    # Plot this cluster in color
    ax.scatter(
        spatial_coords[mask, 0],
        spatial_coords[mask, 1],
        c=cluster_colors[cluster],
        s=spot_size,
        alpha=1.0
    )

    ax.set_title(f'Leiden {cluster}')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')

# Remove empty subplots
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
#plt.savefig('individual_clusters_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Two separate heatmaps: marker gene expression across cell types
# - Titles: "2T  GW24 (bgs4)" and "3T  GW35 (bgs5)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + separate cell type lists for each dataset

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',      # Direct MSN marker
    'DRD2',      # Indirect MSN marker

    'BACH2',     # Transcription factor (DRD1 subtype)
    'EPHA4',     # Eph receptor (DRD1/DRD2 subtype)
    'PENK',      # Enkephalin - classic D2-MSN marker (indirect pathway)
    'GAD1',      # GABAergic interneuron marker

    'PAX6',
    #'PDGFRA',
    'OLIG1',
    #'SOX2',
    #'NES',

    'OLIG2',     # Oligodendrocyte lineage
    'SOX10',     # Oligodendrocyte lineage
    'PDGFRA',     # Strong OPC and pericyte marker (careful overlap with PC)
    'MBP',        # Myelin basic protein - mature oligodendrocyte marker

    'ALDH1L1',   # Astrocyte marker
    'GFAP',      # Astrocyte marker
    'AQP4',      # Astrocyte marker
    'EGFR',
    'ESAM',      # Endothelial marker
    'FN1',       # Endothelial / pericyte
    'LUM',        # Lumican - top VLMC/fibroblast-like marker
    'DCN',        # Decorin - VLMC marker

    'P2RY12',    # Microglia marker
    'CX3CR1',     # Fractalkine receptor - microglia marker (complements P2RY12)
    #'SLC17A7',    # VGLUT1 - excitatory neuron marker (useful for any glutamatergic contamination or comparison)

]

# 2. Explicitly define cell types to KEEP for EACH dataset
# Edit these lists independently - only types in the list will appear in that heatmap
cell_types_bgs4 = [          # 2T  GW24
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',

    'GPC',
    'OPC',
    #'PAX6-TCx',
    'Astro',

    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

cell_types_bgs5 = [          # 3T  GW35
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',

    'GPC',
    'ODC',
    'OPC',
    'Astro',

    #'PVALB',
    #'SST',
    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

# 3. Filtering and area-based QC
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

# for adata in [bgs4_f, bgs5_f]:
#     nuc_area = adata.obs['NucArea'] * (um_per_pixel ** 2)
#     cell_area = adata.obs['Area'] * (um_per_pixel ** 2)
#     mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
#     adata = adata[mask].copy()

# 4. Restrict to desired cell types for each dataset separately
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(cell_types_bgs5)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"{adata.uns.get('name','adata')} - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values
    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0)
    pb_z = pb_z.clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk for each dataset
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order)

# Use only genes present in BOTH datasets (for visual alignment)
shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[:, shared_genes]
pb5 = pb5.loc[:, shared_genes]

# 6b. Reorder cell types in pseudobulk according to custom order
pb4 = pb4.reindex(cell_types_bgs4)
pb5 = pb5.reindex(cell_types_bgs5)

# 7. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).fillna(0).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).fillna(0).astype(int)

# 8. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 6))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top: 2T  GW24
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(pb4, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax1)
ax1.set_title("2T  GW24 (bgs4) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax1.set_ylabel("Cell Type", fontsize=13)
ax1.set_xlabel("")
ax1.tick_params(axis='x', rotation=90, labelsize=11)
cbar1 = g1.collections[0].colorbar
cbar1.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax1.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# Bottom: 3T  GW35
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(pb5, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax2)
ax2.set_title("3T  GW35 (bgs5) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax2.set_ylabel("Cell Type", fontsize=13)
ax2.set_xlabel("Genes", fontsize=13)
ax2.tick_params(axis='x', rotation=90, labelsize=11)
cbar2 = g2.collections[0].colorbar
cbar2.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax2.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

plt.suptitle("\nMarker Gene Expression Across Selected Cell Types\n2T  GW24 (top) vs 3T  GW35 (bottom)",
             fontsize=17, y=0.985)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
# downsampling to 0.33

bgs4 = bgs4[np.random.choice(bgs4.n_obs, int(bgs4.n_obs * 0.33), replace=False), :].copy()
bgs4

In [ ]:
# downsampling to 0.33

bgs5 = bgs5[np.random.choice(bgs5.n_obs, int(bgs5.n_obs * 0.5), replace=False), :].copy()
bgs5

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028  # pixel  µm conversion
min_nuc_area_um2 = 10

# Cell types to include (common cell types or your choice)
cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'eMSN',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]

# 1. Filter datasets by nuclear area
bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

for adata in [bgs4_f, bgs5_f]:
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    adata._inplace_subset_obs(mask)

# 2. Identify common cell types with >10 cells
counts4 = bgs4_f.obs[label_col].value_counts()
counts5 = bgs5_f.obs[label_col].value_counts()

common_candidates = set(cell_types_bgs4) & set(cell_types_bgs5)
common_types = [
    ct for ct in common_candidates
    if ct in counts4.index and counts4[ct] > 10
    and ct in counts5.index and counts5[ct] > 10
]
common_types = sorted(common_types)
print(f"Common cell types for comparison: {common_types}")

# Restrict to common cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(common_types)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(common_types)].copy()

# 3. Compute transcript density
def make_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    df = pd.DataFrame({
        'cell_type': adata.obs[label_col],
        'nCount_RNA': adata.obs['nCount_RNA'],
        'transcript_density': adata.obs['nCount_RNA'] / nuc_area_um2,
        'Dataset': stage
    })
    df['cell_type'] = pd.Categorical(df['cell_type'], categories=common_types, ordered=True)
    return df

df4 = make_df(bgs4_f, 'GW24')
df5 = make_df(bgs5_f, 'GW35')

# 4. Plotting (fixed color issue)
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

plot_params = [
    ('nCount_RNA', 'Median RNA Counts', df4),
    ('nCount_RNA', 'Median RNA Counts', df5),
    ('transcript_density', 'Median Transcript Density', df4),
    ('transcript_density', 'Median Transcript Density', df5)
]

stage_colors = {'GW24': '#1f77b4', 'GW35': '#ff7f0e'}

for i, (y_col, title, df) in enumerate(plot_params):
    ax = axes[i]
    stage = df['Dataset'].iloc[0]
    sns.barplot(
        data=df,
        y='cell_type',
        x=y_col,
        order=common_types,
        color=stage_colors[stage],  # <-- use color instead of palette
        estimator=np.median,
        errorbar=('ci', 95),
        errwidth=2,
        capsize=0.1,
        ax=ax
    )
    ax.set_title(f"{title} ({stage})", fontsize=20)
    ax.set_xlabel(title)
    ax.set_ylabel('Cell Type')
    ax.set_xlim(0, None)

plt.tight_layout()
plt.show()

# 5. Summary table
summary = pd.concat([df4, df5])
medians = summary.groupby(['Dataset', 'cell_type'])[['nCount_RNA', 'transcript_density']].median().reset_index()
summary_table = medians.pivot(index='cell_type', columns='Dataset', values=['nCount_RNA', 'transcript_density'])
summary_table.columns = [f"{metric}_{stage}" for metric, stage in summary_table.columns]
summary_table = summary_table.round(1)
summary_table = summary_table.sort_index()
print("\nSummary Table (Median RNA Counts & Transcript Density):")
print(summary_table)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028  # pixel  µm conversion
min_nuc_area_um2 = 10

# Cell types to include (common cell types or your choice)
cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'eMSN',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]

# 1. Filter datasets by nuclear area
bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

for adata in [bgs4_f, bgs5_f]:
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    adata._inplace_subset_obs(mask)

# 2. Identify common cell types with >10 cells
counts4 = bgs4_f.obs[label_col].value_counts()
counts5 = bgs5_f.obs[label_col].value_counts()

common_candidates = set(cell_types_bgs4) & set(cell_types_bgs5)
common_types = [
    ct for ct in common_candidates
    if ct in counts4.index and counts4[ct] > 10
    and ct in counts5.index and counts5[ct] > 10
]
common_types = sorted(common_types)
print(f"Common cell types for comparison: {common_types}")

# Restrict to common cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(common_types)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(common_types)].copy()

# 3. Compute transcript density
def make_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    df = pd.DataFrame({
        'cell_type': adata.obs[label_col],
        'nCount_RNA': adata.obs['nCount_RNA'],
        'transcript_density': adata.obs['nCount_RNA'] / nuc_area_um2,
        'Dataset': stage
    })
    df['cell_type'] = pd.Categorical(df['cell_type'], categories=common_types, ordered=True)
    return df

df4 = make_df(bgs4_f, 'GW24')
df5 = make_df(bgs5_f, 'GW35')

# 4. Plotting (sorted by nuclei size for transcript density)
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

plot_params = [
    ('nCount_RNA', 'Median RNA Counts', df4, False),
    ('nCount_RNA', 'Median RNA Counts', df5, False),
    ('transcript_density', 'Median Transcript Density', df4, True),
    ('transcript_density', 'Median Transcript Density', df5, True)
]

stage_colors = {'GW24': '#1f77b4', 'GW35': '#ff7f0e'}

for i, (y_col, title, df, sort_by_nuclei) in enumerate(plot_params):
    ax = axes[i]
    stage = df['Dataset'].iloc[0]

    if sort_by_nuclei:
        # Compute median nuclear area per cell type
        nuc_area_um2 = (bgs4_f.obs['NucArea'] * (um_per_pixel ** 2)) if stage == 'GW24' else (bgs5_f.obs['NucArea'] * (um_per_pixel ** 2))
        nuc_median = pd.DataFrame({
            'cell_type': df['cell_type'],
            'nuc_area_um2': nuc_area_um2
        }).groupby('cell_type').median().sort_values('nuc_area_um2', ascending=True)  # smallest nuclei first
        sorted_cell_types = nuc_median.index.tolist()
    else:
        sorted_cell_types = common_types  # keep alphabetical for RNA count plots

    sns.barplot(
        data=df,
        y='cell_type',
        x=y_col,
        order=sorted_cell_types,
        color=stage_colors[stage],
        estimator=np.median,
        errorbar=('ci', 95),
        errwidth=2,
        capsize=0.1,
        ax=ax
    )
    ax.set_title(f"{title} ({stage})", fontsize=20)
    ax.set_xlabel(title)
    ax.set_ylabel('Cell Type')
    ax.set_xlim(0, None)

plt.tight_layout()
plt.show()

# 5. Summary table
summary = pd.concat([df4, df5])
medians = summary.groupby(['Dataset', 'cell_type'])[['nCount_RNA', 'transcript_density']].median().reset_index()
summary_table = medians.pivot(index='cell_type', columns='Dataset', values=['nCount_RNA', 'transcript_density'])
summary_table.columns = [f"{metric}_{stage}" for metric, stage in summary_table.columns]
summary_table = summary_table.round(1)
summary_table = summary_table.sort_index()
print("\nSummary Table (Median RNA Counts & Transcript Density):")
print(summary_table)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028  # pixel  µm conversion
min_nuc_area_um2 = 10

cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'eMSN',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]

# 1. Filter datasets by nuclear area
bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

for adata in [bgs4_f, bgs5_f]:
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    adata._inplace_subset_obs(mask)

# 2. Identify common cell types with >10 cells
counts4 = bgs4_f.obs[label_col].value_counts()
counts5 = bgs5_f.obs[label_col].value_counts()

common_candidates = set(cell_types_bgs4) & set(cell_types_bgs5)
common_types = [
    ct for ct in common_candidates
    if ct in counts4.index and counts4[ct] > 10
    and ct in counts5.index and counts5[ct] > 10
]

# Keep alphabetical base list (used for categorical safety),
# but we'll compute per-stage plotting order from nuclei size.
common_types = sorted(common_types)
print(f"Common cell types for comparison: {common_types}")

bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(common_types)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(common_types)].copy()

# 3. Compute transcript density + also carry nuc_area for sorting
def make_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'nuc_area_um2': nuc_area_um2.values,
        'transcript_density': (adata.obs['nCount_RNA'].values / nuc_area_um2.values),
        'Dataset': stage
    })
    # keep a stable categorical definition; plotting order will be passed via `order=...`
    df['cell_type'] = pd.Categorical(df['cell_type'], categories=common_types, ordered=True)
    return df

df4 = make_df(bgs4_f, 'GW24')
df5 = make_df(bgs5_f, 'GW35')

# 4. Plotting (ALL plots sorted by smallest->largest nuclei per stage)
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

plot_params = [
    ('nCount_RNA', 'Median RNA Counts', df4),
    ('nCount_RNA', 'Median RNA Counts', df5),
    ('transcript_density', 'Median Transcript Density', df4),
    ('transcript_density', 'Median Transcript Density', df5),
]

stage_colors = {'GW24': '#1f77b4', 'GW35': '#ff7f0e'}

for i, (y_col, title, df) in enumerate(plot_params):
    ax = axes[i]
    stage = df['Dataset'].iloc[0]

    # Per-stage cell-type order by median nuclei area (smallest -> largest)
    nuc_median = (
        df.groupby('cell_type', observed=True)['nuc_area_um2']
          .median()
          .sort_values(ascending=True)
    )
    sorted_cell_types = nuc_median.index.tolist()

    sns.barplot(
        data=df,
        y='cell_type',
        x=y_col,
        order=sorted_cell_types,
        color=stage_colors[stage],
        estimator=np.median,
        errorbar=('ci', 95),
        errwidth=2,
        capsize=0.1,
        ax=ax
    )
    ax.set_title(f"{title} ({stage})", fontsize=20)
    ax.set_xlabel(title)
    ax.set_ylabel('Cell Type')
    ax.set_xlim(0, None)

plt.tight_layout()
plt.show()

# 5. Summary table
summary = pd.concat([df4, df5], ignore_index=True)
medians = summary.groupby(['Dataset', 'cell_type'], observed=True)[['nCount_RNA', 'transcript_density']].median().reset_index()
summary_table = medians.pivot(index='cell_type', columns='Dataset', values=['nCount_RNA', 'transcript_density'])
summary_table.columns = [f"{metric}_{stage}" for metric, stage in summary_table.columns]
summary_table = summary_table.round(1).sort_index()

print("\nSummary Table (Median RNA Counts & Transcript Density):")
print(summary_table)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028  # pixel  µm conversion
min_nuc_area_um2 = 10

# Cell types to include (common cell types or your choice)
cell_types_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'eMSN',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_types_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4', 'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]

# 1. Filter datasets by nuclear area
bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

for adata in [bgs4_f, bgs5_f]:
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    adata._inplace_subset_obs(mask)

# 2. Identify common cell types with >10 cells
counts4 = bgs4_f.obs[label_col].value_counts()
counts5 = bgs5_f.obs[label_col].value_counts()

common_candidates = set(cell_types_bgs4) & set(cell_types_bgs5)
common_types = [
    ct for ct in common_candidates
    if ct in counts4.index and counts4[ct] > 10
    and ct in counts5.index and counts5[ct] > 10
]
common_types = sorted(common_types)
print(f"Common cell types for comparison: {common_types}")

# Restrict to common cell types
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(common_types)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(common_types)].copy()

# 3. Compute transcript density
def make_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    df = pd.DataFrame({
        'cell_type': adata.obs[label_col],
        'nCount_RNA': adata.obs['nCount_RNA'],
        'transcript_density': adata.obs['nCount_RNA'] / nuc_area_um2,
        'Dataset': stage
    })
    df['cell_type'] = pd.Categorical(df['cell_type'], categories=common_types, ordered=True)
    return df

df4 = make_df(bgs4_f, 'GW24')
df5 = make_df(bgs5_f, 'GW35')

# 4. Plotting (sorted by nuclei size for transcript density)
fig, axes = plt.subplots(1, 2, figsize=(20, 14))
axes = axes.flatten()

plot_params = [
    #('nCount_RNA', 'Median RNA Counts', df4, False),
    #('nCount_RNA', 'Median RNA Counts', df5, False),
    ('transcript_density', 'Median Transcript Density', df4, True),
    ('transcript_density', 'Median Transcript Density', df5, True)
]

stage_colors = {'GW24': '#1f77b4', 'GW35': '#ff7f0e'}

for i, (y_col, title, df, sort_by_nuclei) in enumerate(plot_params):
    ax = axes[i]
    stage = df['Dataset'].iloc[0]

    if sort_by_nuclei:
        # Compute median nuclear area per cell type
        nuc_area_um2 = (bgs4_f.obs['NucArea'] * (um_per_pixel ** 2)) if stage == 'GW24' else (bgs5_f.obs['NucArea'] * (um_per_pixel ** 2))
        nuc_median = pd.DataFrame({
            'cell_type': df['cell_type'],
            'nuc_area_um2': nuc_area_um2
        }).groupby('cell_type').median().sort_values('nuc_area_um2', ascending=True)  # smallest nuclei first
        sorted_cell_types = nuc_median.index.tolist()
    else:
        sorted_cell_types = common_types  # keep alphabetical for RNA count plots

    sns.barplot(
        data=df,
        y='cell_type',
        x=y_col,
        order=sorted_cell_types,
        color=stage_colors[stage],
        estimator=np.median,
        errorbar=('ci', 95),
        errwidth=2,
        capsize=0.1,
        ax=ax
    )
    ax.set_title(f"{title} ({stage})", fontsize=20)
    ax.set_xlabel(title)
    ax.set_ylabel('Cell Type')
    ax.set_xlim(0, None)

plt.tight_layout()
plt.show()

# 5. Summary table
summary = pd.concat([df4, df5])
medians = summary.groupby(['Dataset', 'cell_type'])[['nCount_RNA', 'transcript_density']].median().reset_index()
summary_table = medians.pivot(index='cell_type', columns='Dataset', values=['nCount_RNA', 'transcript_density'])
summary_table.columns = [f"{metric}_{stage}" for metric, stage in summary_table.columns]
summary_table = summary_table.round(1)
summary_table = summary_table.sort_index()
print("\nSummary Table (Median RNA Counts & Transcript Density):")
print(summary_table)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10

# Filter by nuclear area
def filter_and_make_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    ad = adata[mask].copy()

    return pd.DataFrame({
        'cell_type': ad.obs[label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

df4 = filter_and_make_df(bgs4, 'GW24')
df5 = filter_and_make_df(bgs5, 'GW35')

# Keep common cell types with enough cells
min_cells = 30
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in set(cts4.index) & set(cts5.index)
    if cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

# Color palette (consistent across time points)
palette = dict(zip(
    common_types,
    sns.color_palette('tab20', len(common_types))
))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

# GW24 (Left)
for ct in common_types:
    sns.kdeplot(
        data=df4[df4['cell_type'] == ct],
        x='nuc_area_um2',
        ax=axes[0],
        color=palette[ct],
        lw=2,
        alpha=0.8
    )

axes[0].set_title('GW24 (Second Trimester)')
axes[0].set_xlabel('Nuclear Area (µm²)')
axes[0].set_ylabel('Density')

# GW35 (Right)
for ct in common_types:
    sns.kdeplot(
        data=df5[df5['cell_type'] == ct],
        x='nuc_area_um2',
        ax=axes[1],
        color=palette[ct],
        lw=2,
        alpha=0.8
    )

axes[1].set_title('GW35 (Third Trimester)')
axes[1].set_xlabel('Nuclear Area (µm²)')

# Label cell types on the RIGHT side
y_max = axes[1].get_ylim()[1]
x_max = axes[1].get_xlim()[1]

for i, ct in enumerate(common_types):
    axes[1].text(
        x_max * 1.01,
        y_max * (0.95 - i * 0.04),
        ct,
        color=palette[ct],
        fontsize=12,
        va='top'
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Helper: build nuclear size dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2

    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')

# Restrict to cell types starting with "D"
df4 = df4[df4['cell_type'].str.startswith('D')]
df5 = df5[df5['cell_type'].str.startswith('D')]

# Keep only cell types present at BOTH timepoints
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in set(cts4.index) & set(cts5.index)
    if cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

print(f"Included D* cell types: {common_types}")

# Compute median nuclear size per cell type and stage
summary = (
    pd.concat([df4, df5])
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .median()
    .reset_index()
)

# Ensure correct time ordering
summary['stage'] = pd.Categorical(
    summary['stage'],
    categories=['GW24', 'GW35'],
    ordered=True
)

# Plot: two-point time series
fig, ax = plt.subplots(figsize=(10, 7))

# Manual color control
# DRD1 = blues, DRD2 = reds
palette = {
    # DRD1
    'DRD1-BACH2': '#27ab44',                 # blue
    'DRD1-EPHA4': '#08306b',                 # lighter blue
    'DRD1-eccentric-CASZ1': '#10c4e8',       # dark blue

    # DRD2
    'DRD2-BACH2': '#929494',                 # red
    'DRD2-EPHA4': '#a15747',                 # light red
    #'DRD2-eccentric-CASZ1': '#7f0000',       # dark red
}

# Manual label offsets (µm²)
# Positive = move label UP, Negative = move label DOWN
label_offsets = {
    # Example (edit freely):
    'DRD1-BACH2': -0.5,
    'DRD1-EPHA4': +0.5,
    # 'DRD1-eccentric-CASZ1': 10,
    # 'DRD2-BACH2': -6,
}

for ct in common_types:
    sub = summary[summary['cell_type'] == ct]

    ax.plot(
        sub['stage'],
        sub['nuc_area_um2'],
        marker='o',
        lw=2.5,
        color=palette[ct]
    )

    # Label on the right (GW35) with manual offset
    y_end = sub[sub['stage'] == 'GW35']['nuc_area_um2'].values[0]
    y_offset = label_offsets.get(ct, 0)

    ax.text(
        1.02,
        y_end + y_offset,
        ct,
        color=palette[ct],
        fontsize=13,
        va='center',
        transform=ax.get_yaxis_transform()
    )

# Axis formatting
ax.set_title('Nuclear Size Changes Across Development')
ax.set_xlabel('Developmental Stage')
ax.set_ylabel('Median Nuclear Area (µm²)')
ax.set_xlim(-0.1, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Helper: build nuclear transcript density dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2

    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'nCount_RNA': adata.obs.loc[mask, 'nCount_RNA'],
        'transcript_density': adata.obs.loc[mask, 'nCount_RNA'] / nuc_area_um2[mask],
        'stage': stage
    })

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')

# Restrict to cell types starting with "D"
df4 = df4[df4['cell_type'].str.startswith('D')]
df5 = df5[df5['cell_type'].str.startswith('D')]

# Keep only cell types present at BOTH timepoints
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in set(cts4.index) & set(cts5.index)
    if cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

print(f"Included D* cell types: {common_types}")

# Compute median transcript density per cell type and stage
summary = (
    pd.concat([df4, df5])
    .groupby(['cell_type', 'stage'])['transcript_density']
    .median()
    .reset_index()
)

summary['stage'] = pd.Categorical(
    summary['stage'],
    categories=['GW24', 'GW35'],
    ordered=True
)

# Plot: two-point time series
fig, ax = plt.subplots(figsize=(10, 7))

# Manual color control (unchanged)
palette = {
    # DRD1
    'DRD1-BACH2': '#27ab44',
    'DRD1-EPHA4': '#08306b',
    'DRD1-eccentric-CASZ1': '#10c4e8',

    # DRD2
    'DRD2-BACH2': '#929494',
    'DRD2-EPHA4': '#a15747',
}

# Fallback safety
for ct in common_types:
    if ct not in palette:
        palette[ct] = '#7f7f7f'

# Manual label offsets (density units)
label_offsets = {
    'DRD1-BACH2': 0.5,
    'DRD2-EPHA4': +1,
}

# Draw lines + labels
for ct in common_types:
    sub = summary[summary['cell_type'] == ct]

    ax.plot(
        sub['stage'],
        sub['transcript_density'],
        marker='o',
        lw=2.5,
        color=palette[ct]
    )

    # Right-side label (GW35)
    y_end = sub[sub['stage'] == 'GW35']['transcript_density'].values[0]
    y_offset = label_offsets.get(ct, 0)

    ax.text(
        1.02,
        y_end + y_offset,
        ct,
        color=palette[ct],
        fontsize=13,
        va='center',
        transform=ax.get_yaxis_transform()
    )

# Axis formatting
ax.set_title('Nuclear Transcript Density Changes Across Development')
ax.set_xlabel('Developmental Stage')
ax.set_ylabel('Median Nuclear Transcript Density\n(transcripts / µm²)')
ax.set_xlim(-0.1, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Very large font settings (match your screenshot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

groups = {
    'DRD subtypes': ['DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
                     'DRD2-BACH2', 'DRD2-EPHA4'],
    'Glia': ['OPC', 'GPC', 'Astro', 'MGC-1'],
    'Vascular': ['EC', 'PC']
}

palettes = {
    'DRD subtypes': {
        'DRD1-BACH2': '#27ab44',
        'DRD1-EPHA4': '#08306b',
        'DRD1-eccentric-CASZ1': '#10c4e8',
        'DRD2-BACH2': '#d95f02',
        'DRD2-EPHA4': '#a15747',
    },
    'Glia': {
        'OPC':    '#d95f02',
        'GPC':    '#1b9e77',
        'Astro':  '#7570b3',
        'MGC-1':  '#e7298a',
    },
    'Vascular': {
        'EC': '#e41a1c',
        'PC': '#4daf4a',
    }
}

color_map = {ct: col for g in palettes.values() for ct, col in g.items()}

# Helper: build per-cell dataframe (area + counts)
def build_cell_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    desired = set(sum(groups.values(), []))
    df = df[df['cell_type'].isin(desired)].copy()

    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24')
df5_cells = build_cell_df(bgs5, 'GW35')

# Helper: collapse to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    summary = (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )
    return summary

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two separate scatter plots (GW24 and GW35)
def scatter_plot(summary_df, stage, ax):
    dot_size = 70          # <-- a bit bigger
    title_fs = 18          # <-- smaller title font
    label_dx = 0.35        # <-- shift right (in µm² units)
    label_dy = 25          # <-- shift up (in nCount_RNA units)

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_area_um2']
        y = row['median_nCount_RNA']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=16,
            fontweight='bold',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Median nCount_RNA vs Median Nuclear Area",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0])
scatter_plot(sum5, "GW35 (bgs5)", axes[1])

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Very large font settings (match your screenshot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

groups = {
    'DRD subtypes': ['DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
                     'DRD2-BACH2', 'DRD2-EPHA4'],
    'Glia': ['OPC', 'GPC', 'Astro', 'MGC-1'],
    'Vascular': ['EC', 'PC']
}

palettes = {
    'DRD subtypes': {
        'DRD1-BACH2': '#27ab44',
        'DRD1-EPHA4': '#08306b',
        'DRD1-eccentric-CASZ1': '#10c4e8',
        'DRD2-BACH2': '#d95f02',
        'DRD2-EPHA4': '#a15747',
    },
    'Glia': {
        'OPC':    '#d95f02',
        'GPC':    '#1b9e77',
        'Astro':  '#7570b3',
        'MGC-1':  '#e7298a',
    },
    'Vascular': {
        'EC': '#e41a1c',
        'PC': '#4daf4a',
    }
}
color_map = {ct: col for g in palettes.values() for ct, col in g.items()}

# Geometry helper: sphere volume from 2D area
# A = pi r^2  -> r = sqrt(A/pi)
# V = 4/3 pi r^3
def sphere_volume_from_area(area_um2: np.ndarray) -> np.ndarray:
    r = np.sqrt(area_um2 / np.pi)
    v = (4/3) * np.pi * (r ** 3)
    return v  # µm^3

# Helper: build per-cell dataframe (volume + counts)
def build_cell_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    # rename Astros -> Astro (same as your other script)
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by nucleus size threshold
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only your desired cell types (those in groups)
    desired = set(sum(groups.values(), []))
    df = df[df['cell_type'].isin(desired)].copy()

    # estimate nuclear volume per cell
    df['nuc_vol_um3'] = sphere_volume_from_area(df['nuc_area_um2'].to_numpy())

    # enforce min_cells per cell type (within this stage)
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24')
df5_cells = build_cell_df(bgs5, 'GW35')

# Collapse to per-cell-type medians for scatter points
#   x = median nuclear volume, y = median transcript count
def summarize_for_scatter(df_cells):
    summary = (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_vol_um3=('nuc_vol_um3', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )
    return summary

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two separate scatter plots (GW24 and GW35)
def scatter_plot(summary_df, stage, ax):
    dot_size = 70       # dots a bit bigger
    title_fs = 18       # smaller title font

    # label offsets (now x-axis is volume; pick small relative offsets)
    label_dx = 40       # µm^3
    label_dy = 25       # nCount_RNA

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_vol_um3']
        y = row['median_nCount_RNA']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=16,
            fontweight='bold',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Median nCount_RNA vs Estimated Nuclear Volume",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Estimated Nuclear Volume (µm³) [sphere from 2D area]")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0])
scatter_plot(sum5, "GW35 (bgs5)", axes[1])

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_vol_um3').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_vol_um3').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Very large font settings (match your screenshot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# NEW: include your added cell types (stage-specific lists)
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]
cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

# Use your explicit colors (fallback to gray if missing)
cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}
cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Geometry helper: sphere volume from 2D area
# A = pi r^2  -> r = sqrt(A/pi)
# V = 4/3 pi r^3
def sphere_volume_from_area(area_um2: np.ndarray) -> np.ndarray:
    r = np.sqrt(area_um2 / np.pi)
    v = (4/3) * np.pi * (r ** 3)
    return v  # µm^3

# Helper: build per-cell dataframe (volume + counts) (stage-specific types)
def build_cell_df(adata, stage, allowed_types):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    # rename Astros -> Astro
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by nucleus size threshold
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired cell types for this dataset
    df = df[df['cell_type'].isin(allowed_types)].copy()

    # estimate nuclear volume per cell
    df['nuc_vol_um3'] = sphere_volume_from_area(df['nuc_area_um2'].to_numpy())

    # enforce min_cells per cell type (within this stage)
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Collapse to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_vol_um3=('nuc_vol_um3', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two separate scatter plots (GW24 and GW35)
# (labels moved back closer to dots)
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18

    # bring labels closer than your previous version
    label_dx = 5    # µm^3 (smaller than 40)
    label_dy = -20    # counts (smaller than 25)

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_vol_um3']
        y = row['median_nCount_RNA']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=14,
            fontweight='normal',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Median nCount_RNA vs Estimated Nuclear Volume",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Estimated Nuclear Volume (µm³) [sphere from 2D area]")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_vol_um3').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_vol_um3').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Cell types + colors (new cell types included)
cell_type_order_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'eMSN', 'eMGE',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Geometry helper: sphere volume from 2D area
# A = pi r^2  -> r = sqrt(A/pi)
# V = 4/3 pi r^3
def sphere_volume_from_area(area_um2: np.ndarray) -> np.ndarray:
    r = np.sqrt(area_um2 / np.pi)
    v = (4/3) * np.pi * (r ** 3)
    return v  # µm^3

# Build per-cell dataframe (volume + transcript density)
def build_cell_df(adata, stage, allowed_types):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by nucleus size threshold
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types for this dataset
    df = df[df['cell_type'].isin(allowed_types)].copy()

    # transcript density per cell
    df['transcript_density'] = df['nCount_RNA'] / df['nuc_area_um2']

    # estimated nuclear volume per cell
    df['nuc_vol_um3'] = sphere_volume_from_area(df['nuc_area_um2'].to_numpy())

    # enforce min_cells per cell type
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4 = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5 = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize per cell type (medians) for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_vol_um3=('nuc_vol_um3', 'median'),
            median_transcript_density=('transcript_density', 'median'),
            n_cells=('transcript_density', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4)
sum5 = summarize_for_scatter(df5)

# Plot: transcript density (Y) vs nuclear volume (X)
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18

    # labels close-ish to dots
    label_dx = 15     # µm^3
    label_dy = 0.001 # density units (counts/µm²)

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_vol_um3']
        y = row['median_transcript_density']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=14,
            fontweight='normal',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Transcript Density vs Estimated Nuclear Volume",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Estimated Nuclear Volume (µm³) [sphere from 2D area]")
    ax.set_ylabel("Median Transcript Density (nCount_RNA / µm²)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_vol_um3').round(6))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_vol_um3').round(6))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit orders + colors (new cell types included)
cell_type_order_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'eMSN', 'eMGE',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Geometry helper: sphere volume from 2D area
# A = pi r^2  -> r = sqrt(A/pi)
# V = 4/3 pi r^3
def sphere_volume_from_area(area_um2: np.ndarray) -> np.ndarray:
    r = np.sqrt(area_um2 / np.pi)
    v = (4/3) * np.pi * (r ** 3)
    return v  # µm^3

# Build per-cell dataframe with nuclear VOLUME + transcript density
# density = nCount_RNA / nuc_area_um2  (same density definition as before)
def build_cell_df_volume(adata, stage, allowed_types):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by nuclear size
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # restrict to chosen cell types for this dataset
    df = df[df['cell_type'].isin(allowed_types)].copy()

    # per-cell density (same as your area-residual code)
    df['transcript_density'] = df['nCount_RNA'] / df['nuc_area_um2']

    # per-cell estimated nuclear volume
    df['nuc_vol_um3'] = sphere_volume_from_area(df['nuc_area_um2'].to_numpy())

    # enforce minimum cells per type
    vc = df['cell_type'].value_counts()
    keep_types = vc[vc >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4 = build_cell_df_volume(bgs4, 'GW24', cell_type_order_bgs4)
df5 = build_cell_df_volume(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize per cell type + compute residual vs VOLUME trend
# Fit: density ~ volume   (using per-cell-type medians, like before)
def summarize_and_residualize_volume(df):
    summ = (
        df.groupby('cell_type')
          .agg(
              nuc_vol_med=('nuc_vol_um3', 'median'),
              density_med=('transcript_density', 'median'),
              n_cells=('transcript_density', 'size')
          )
          .reset_index()
    )

    x = summ['nuc_vol_med'].to_numpy()
    y = summ['density_med'].to_numpy()
    b, a = np.polyfit(x, y, 1)  # y ≈ b*x + a

    summ['density_fit'] = b * summ['nuc_vol_med'] + a
    summ['density_resid'] = summ['density_med'] - summ['density_fit']

    return summ, (a, b)

sum4, (a4, b4) = summarize_and_residualize_volume(df4)
sum5, (a5, b5) = summarize_and_residualize_volume(df5)

# Plot: residual density ranking (volume-based trend)
def residual_barplot(summ, stage, colors, ax):
    summ2 = summ.sort_values('density_resid', ascending=True).copy()

    title_fs = 15
    label_fs = 12

    # symmetric x limits based on max residual magnitude
    m = np.nanmax(np.abs(summ2['density_resid'].to_numpy()))
    ax.set_xlim(-1.15 * m, 1.15 * m)

    # label offset based on x-range (kept fairly close)
    xspan = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = 0.008 * xspan  # close-ish labels

    ax.axvline(0, lw=2, alpha=0.5)

    for i, row in enumerate(summ2.itertuples(index=False)):
        ct = row.cell_type
        x = row.density_resid

        ax.scatter(
            x, i,
            s=70,
            color=colors.get(ct, 'gray'),
            edgecolor='black',
            linewidth=0.6,
            zorder=5
        )

        ax.text(
            x + (offset if x >= 0 else -offset),
            i,
            ct,
            ha='left' if x >= 0 else 'right',
            va='center',
            fontsize=label_fs,
            fontweight='normal'
        )

    ax.set_yticks([])
    ax.set_xlabel("Transcript density residual (median density − fitted volume-trend)")
    ax.set_title(f"{stage}: density enrichment/depletion vs volume trend",
                 fontsize=title_fs, fontweight='normal')
    ax.grid(axis='x', alpha=0.25, linestyle='--')

# Make figure
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'hspace': 0.35})

residual_barplot(sum4, "GW24 (bgs4)", cell_type_colors_bgs4, axes[0])
residual_barplot(sum5, "GW35 (bgs5)", cell_type_colors_bgs5, axes[1])

plt.tight_layout()
plt.show()

# Optional: print residual tables (volume-based)
print("\nGW24 residual density summary (volume-based; positive = higher than expected for volume):")
print(
    sum4[['cell_type', 'nuc_vol_med', 'density_med', 'density_fit', 'density_resid', 'n_cells']]
    .sort_values('density_resid', ascending=False)
    .round(6)
    .to_string(index=False)
)

print("\nGW35 residual density summary (volume-based; positive = higher than expected for volume):")
print(
    sum5[['cell_type', 'nuc_vol_med', 'density_med', 'density_fit', 'density_resid', 'n_cells']]
    .sort_values('density_resid', ascending=False)
    .round(6)
    .to_string(index=False)
)

In [ ]:
#change bgs4 '?' cell type to 'eMGE'

label_col = 'adjusted_L3_transferred'

bgs4.obs[label_col] = (
    bgs4.obs[label_col]
    .replace({'?': 'eMGE'})
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Very large font settings (match your scatter plot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (area + counts)
def build_cell_df(adata, stage, allowed_order):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    # match your previous behavior
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter small nuclei
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types (based on your explicit order)
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # enforce min_cells per type (within stage)
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two scatter plots (GW24 and GW35) using AREA on x-axis
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18

    # label offsets (same units as axes)
    label_dx = 1   # µm²
    label_dy = -0.9     # counts

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_area_um2']
        y = row['median_nCount_RNA']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=16,
            fontweight='bold',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Median nCount_RNA vs Median Nuclear Area",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

# Optional: print tables used for scatter points
print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Very large font settings (match your scatter plot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (area + counts)
def build_cell_df(adata, stage, allowed_order):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter small nuclei
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # enforce min_cells per type
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: scatter + dashed regression line
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18
    label_fs = 15

    # Use RELATIVE offsets so it always works regardless of scale
    xspan = summary_df['median_nuc_area_um2'].max() - summary_df['median_nuc_area_um2'].min()
    yspan = summary_df['median_nCount_RNA'].max() - summary_df['median_nCount_RNA'].min()

    label_dx = 0.02 * xspan          # ~2% of x-range
    label_dy = -0.02 * yspan         # move DOWN by ~4% of y-range

    # regression line (fit across cell-type medians)
    x = summary_df['median_nuc_area_um2'].to_numpy()
    y = summary_df['median_nCount_RNA'].to_numpy()

    if len(summary_df) >= 2:
        b, a = np.polyfit(x, y, 1)  # y = b*x + a
        xs = np.linspace(x.min(), x.max(), 200)
        ys = b * xs + a
        ax.plot(xs, ys, linestyle='--', linewidth=2.2, alpha=0.8, color='black', zorder=1)

    # scatter + labels
    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x0 = row['median_nuc_area_um2']
        y0 = row['median_nCount_RNA']

        ax.scatter(
            x0, y0,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x0 + label_dx,
            y0 + label_dy,
            ct,
            fontsize=label_fs,
            fontweight='normal',   # <-- no bold
            ha='left',
            va='bottom'
        )

    ax.set_title(
        f"{stage}: Median nCount_RNA vs Median Nuclear Area",
        fontweight='normal',      # <-- no bold
        fontsize=title_fs,
        pad=12
    )
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.15, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

# Optional: print tables used for scatter points
print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Very large font settings (match your scatter plot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (area + counts)
def build_cell_df(adata, stage, allowed_order):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter small nuclei
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # enforce min_cells per type
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: scatter + dashed regression line (to edges) + spine styling
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 250
    title_fs = 18
    label_fs = 16.5  # teeny hint bigger

    # relative offsets so it works across scales
    xspan = summary_df['median_nuc_area_um2'].max() - summary_df['median_nuc_area_um2'].min()
    yspan = summary_df['median_nCount_RNA'].max() - summary_df['median_nCount_RNA'].min()

    label_dx = 0.02 * xspan
    label_dy = -0.02 * yspan   # <-- your requested y offset

    # scatter points + labels
    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x0 = row['median_nuc_area_um2']
        y0 = row['median_nCount_RNA']

        ax.scatter(
            x0, y0,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x0 + label_dx,
            y0 + label_dy,
            ct,
            fontsize=label_fs,
            fontweight='normal',
            ha='left',
            va='bottom'
        )

    # titles/labels (no bold)
    ax.set_title(
        f"{stage}: Median nCount_RNA vs Median Nuclear Area",
        fontweight='normal',
        fontsize=title_fs,
        pad=12
    )
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")

    # grid
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

    # spine styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.0)
    ax.spines['bottom'].set_linewidth(2.0)

    # dashed regression line that goes to the plot edges
    x = summary_df['median_nuc_area_um2'].to_numpy()
    y = summary_df['median_nCount_RNA'].to_numpy()

    if len(summary_df) >= 2:
        b, a = np.polyfit(x, y, 1)  # y = b*x + a

        # Force axis limits first (so line can span full width)
        ax.set_xlim(x.min() - 0.05 * xspan, x.max() + 0.05 * xspan)
        xmin, xmax = ax.get_xlim()

        xs = np.array([xmin, xmax])
        ys = b * xs + a

        ax.plot(
            xs, ys,
            linestyle='--',
            linewidth=2.2,
            alpha=0.8,
            color='black',
            zorder=1
        )

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Illustrator-friendly PDF settings + Arial font
mpl.rcParams.update({
    # keep text editable in Illustrator
    'pdf.fonttype': 42,
    'ps.fonttype': 42,

    # use Arial everywhere (fallbacks included)
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],

    # your large style
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (area + counts)
def build_cell_df(adata, stage, allowed_order):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter small nuclei
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # enforce min_cells per type
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: scatter + dashed regression line (to edges) + spine styling
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 250
    title_fs = 18
    label_fs = 16.5

    # relative offsets so it works across scales
    xspan = summary_df['median_nuc_area_um2'].max() - summary_df['median_nuc_area_um2'].min()
    yspan = summary_df['median_nCount_RNA'].max() - summary_df['median_nCount_RNA'].min()

    label_dx = 0.02 * xspan
    label_dy = -0.02 * yspan

    # scatter points + labels
    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x0 = row['median_nuc_area_um2']
        y0 = row['median_nCount_RNA']

        ax.scatter(
            x0, y0,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x0 + label_dx,
            y0 + label_dy,
            ct,
            fontsize=label_fs,
            fontweight='normal',
            ha='left',
            va='bottom'
        )

    # titles/labels (no bold)
    ax.set_title(
        f"{stage}: Median nCount_RNA vs Median Nuclear Area",
        fontweight='normal',
        fontsize=title_fs,
        pad=12
    )
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")

    # grid
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

    # spine styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.0)
    ax.spines['bottom'].set_linewidth(2.0)

    # dashed regression line that goes to the plot edges
    x = summary_df['median_nuc_area_um2'].to_numpy()
    y = summary_df['median_nCount_RNA'].to_numpy()

    if len(summary_df) >= 2:
        b, a = np.polyfit(x, y, 1)  # y = b*x + a

        ax.set_xlim(x.min() - 0.05 * xspan, x.max() + 0.05 * xspan)
        xmin, xmax = ax.get_xlim()

        xs = np.array([xmin, xmax])
        ys = b * xs + a

        ax.plot(
            xs, ys,
            linestyle='--',
            linewidth=2.2,
            alpha=0.8,
            color='black',
            zorder=1
        )

# Make figure + save PDF (editable text)
fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()

out_pdf = "nCountRNA_vs_NuclearArea_GW24_GW35_editable.pdf"
plt.savefig(out_pdf, format="pdf", dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved PDF: {out_pdf}")

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(2))

In [ ]:
from scipy.stats import pearsonr, spearmanr

# Correlation: median nuclear area vs median nCount_RNA (per cell type)
def corr_stats(summary_df, stage_name):
    x = summary_df['median_nuc_area_um2'].to_numpy()
    y = summary_df['median_nCount_RNA'].to_numpy()

    r, p_r = pearsonr(x, y)
    rho, p_rho = spearmanr(x, y)

    print(f"\nCorrelation ({stage_name})  [per cell-type medians]")
    print(f"  Pearson r   = {r:.3f}   (p = {p_r:.3e})")
    print(f"  Spearman ρ  = {rho:.3f} (p = {p_rho:.3e})")
    print(f"  n cell types = {len(summary_df)}")

corr_stats(sum4, "GW24 (bgs4)")
corr_stats(sum5, "GW35 (bgs5)")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Very large font settings (match your screenshot style)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_cell_area_um2 = 10   # same threshold logic, now applied to cell Area
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (CELL area + counts)
#   NOTE: 'Area' is in pixels^2, so convert to µm² like NucArea.
def build_cell_df(adata, stage, allowed_order):
    cell_area_um2 = adata.obs['Area'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'cell_area_um2': cell_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    # match your previous behavior
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter tiny cells
    df = df[df['cell_area_um2'] >= min_cell_area_um2].copy()

    # keep only desired types (based on your explicit order)
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # enforce min_cells per type (within stage)
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_cell_area_um2=('cell_area_um2', 'median'),
            median_nCount_RNA=('nCount_RNA', 'median'),
            n_cells=('nCount_RNA', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two scatter plots (GW24 and GW35) using CELL AREA on x-axis
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18

    # label offsets (same units as axes)
    label_dx = 0.6   # µm² (cell areas can be larger than nuclei, bump slightly)
    label_dy = 25    # counts

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_cell_area_um2']
        y = row['median_nCount_RNA']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=16,
            fontweight='bold',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Median nCount_RNA vs Median Cell Area",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Median Cell Area (µm²) (not nuclear)")
    ax.set_ylabel("Median Transcript Count (nCount_RNA)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_cell_area_um2').round(2))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_cell_area_um2').round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit cell type orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-cell dataframe (NUCLEAR area + transcript density)
def build_cell_df(adata, stage, allowed_order):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    # rename Astros -> Astro (if present)
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by min nuclear area
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # keep only desired types
    df = df[df['cell_type'].isin(allowed_order)].copy()

    # transcript density per cell
    df['transcript_density'] = df['nCount_RNA'] / df['nuc_area_um2']

    # enforce min_cells per type
    keep = df['cell_type'].value_counts()
    keep_types = keep[keep >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4_cells = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5_cells = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize to per-cell-type medians for scatter points
#   x = median nuclear area
#   y = median transcript density
def summarize_for_scatter(df_cells):
    return (
        df_cells
        .groupby('cell_type')
        .agg(
            median_nuc_area_um2=('nuc_area_um2', 'median'),
            median_transcript_density=('transcript_density', 'median'),
            n_cells=('transcript_density', 'size')
        )
        .reset_index()
    )

sum4 = summarize_for_scatter(df4_cells)
sum5 = summarize_for_scatter(df5_cells)

# Plotting: two scatter plots (GW24 and GW35)
def scatter_plot(summary_df, stage, ax, color_map):
    dot_size = 70
    title_fs = 18

    # label offsets
    label_dx = 0.35   # µm²
    label_dy = 0.0005 # density units (counts/um²) -> small shift

    for _, row in summary_df.iterrows():
        ct = row['cell_type']
        x = row['median_nuc_area_um2']
        y = row['median_transcript_density']

        ax.scatter(
            x, y,
            s=dot_size,
            color=color_map.get(ct, 'gray'),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7,
            zorder=5
        )

        ax.text(
            x + label_dx,
            y + label_dy,
            ct,
            fontsize=16,
            fontweight='bold',
            ha='left',
            va='bottom'
        )

    ax.set_title(f"{stage}: Transcript Density vs Nuclear Area",
                 fontweight='bold', fontsize=title_fs, pad=12)
    ax.set_xlabel("Median Nuclear Area (µm²)")
    ax.set_ylabel("Median Transcript Density (nCount_RNA / µm²)")
    ax.grid(alpha=0.25, linestyle='--', linewidth=1.0)

fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=False)

scatter_plot(sum4, "GW24 (bgs4)", axes[0], cell_type_colors_bgs4)
scatter_plot(sum5, "GW35 (bgs5)", axes[1], cell_type_colors_bgs5)

plt.tight_layout()
plt.show()

# Optional: print tables used for scatter points
print("\nGW24 (bgs4) scatter points (per cell type):")
print(sum4.sort_values('median_nuc_area_um2').round(4))

print("\nGW35 (bgs5) scatter points (per cell type):")
print(sum5.sort_values('median_nuc_area_um2').round(4))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Style settings
mpl.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# NEW: handle stages separately (different cell types across bgs4 vs bgs5)
# Use your explicit orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',
    'eMGE',
    'GPC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',
    'GPC',
    'ODC',
    'OPC',
    'Astro',
    'EC',
    'PC',
    'MGC-1',
]

cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Helper: build per-stage dataframe
def build_df_stage(adata, stage, allowed_types):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2

    df = pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col].astype(str),
        'nuc_area_um2': nuc_area_um2[mask].values,
        'stage': stage
    })

    # match your prior rename behavior (safe even if absent)
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # restrict to your desired types for that dataset
    df = df[df['cell_type'].isin(allowed_types)].copy()

    # min-cells filter per stage
    vc = df['cell_type'].value_counts()
    keep = vc[vc >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep)].copy()

    return df

df4 = build_df_stage(bgs4, 'GW24', cell_type_order_bgs4)
df5 = build_df_stage(bgs5, 'GW35', cell_type_order_bgs5)

# combine (still useful for a single summary table)
df_all = pd.concat([df4, df5], ignore_index=True)

# Summary: median + approx 95% CI (normal approx around median)
summary = (
    df_all
    .groupby(['stage', 'cell_type'])['nuc_area_um2']
    .agg(median='median', count='count', std='std')
    .reset_index()
)

summary['sem'] = summary['std'] / np.sqrt(summary['count'])
summary['ci_low']  = summary['median'] - 1.96 * summary['sem']
summary['ci_high'] = summary['median'] + 1.96 * summary['sem']

# MAIN OUTPUT YOU WANTED: print nuclear sizes + CI, stage handled separately
def print_stage_table(summary_df, stage, desired_order):
    sub = summary_df[summary_df['stage'] == stage].copy()
    sub['cell_type'] = pd.Categorical(sub['cell_type'], categories=desired_order, ordered=True)
    sub = sub.sort_values('cell_type')

    out = sub[['cell_type', 'count', 'median', 'ci_low', 'ci_high']].copy()
    out = out.rename(columns={
        'count': 'n_cells',
        'median': f'{stage}_median_um2',
        'ci_low': f'{stage}_ci_low_um2',
        'ci_high': f'{stage}_ci_high_um2'
    })

    # rounding for readability
    num_cols = [c for c in out.columns if c != 'cell_type']
    out[num_cols] = out[num_cols].astype(float).round(2)

    print(f"\nNuclear area summary (median + 95% CI): {stage}")
    print(out.to_string(index=False))

print_stage_table(summary, 'GW24', cell_type_order_bgs4)
print_stage_table(summary, 'GW35', cell_type_order_bgs5)

# OPTIONAL: if you still want a wide merged table (union of cell types)
wide = (
    summary[['stage', 'cell_type', 'count', 'median', 'ci_low', 'ci_high']]
    .pivot(index='cell_type', columns='stage')
)

# flatten columns
wide.columns = [f"{col}_{stage}" for col, stage in wide.columns]
wide = wide.reset_index()

# keep a stable print order: GW24 order first, then any extras from GW35
union_order = cell_type_order_bgs4 + [ct for ct in cell_type_order_bgs5 if ct not in cell_type_order_bgs4]
wide['cell_type'] = pd.Categorical(wide['cell_type'], categories=union_order, ordered=True)
wide = wide.sort_values('cell_type')

# round numeric cols
for c in wide.columns:
    if c != 'cell_type':
        wide[c] = wide[c].astype(float).round(2)

print("\nWide nuclear area summary (union of cell types; median + 95% CI):")
print(wide.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Your explicit orders + colors
cell_type_order_bgs4 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'eMSN', 'eMGE',
    'GPC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs4 = {
    'eMGE': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
    'eMSN': '#6A00FF',
}

cell_type_order_bgs5 = [
    'DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
    'DRD2-BACH2', 'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1', 'CABLES1',
    'GPC', 'ODC', 'OPC', 'Astro',
    'EC', 'PC', 'MGC-1'
]
cell_type_colors_bgs5 = {
    'ODC': '#1f77b4',
    'Astro': '#ff7f0e',
    'DRD1-BACH2': '#279e68',
    'DRD1-EPHA4': '#d62728',
    'DRD1-eccentric-CASZ1': '#aa40fc',
    'DRD2-BACH2': '#8c564b',
    'DRD2-EPHA4': '#e377c2',
    'DRD2-eccentric-CASZ1': '#FF1493',
    'CABLES1': '#7F1D1D',
    'EC': '#00A6A6',
    'GPC': '#17becf',
    'MGC-1': '#aec7e8',
    'OPC': '#ffbb78',
    'PC': '#009E73',
}

# Build per-cell dataframe with nuclear area + transcript density
def build_cell_df(adata, stage, allowed_types):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)

    df = pd.DataFrame({
        'cell_type': adata.obs[label_col].astype(str),
        'nuc_area_um2': nuc_area_um2.values,
        'nCount_RNA': adata.obs['nCount_RNA'].values,
        'stage': stage
    })

    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')

    # filter by nuclear size
    df = df[df['nuc_area_um2'] >= min_nuc_area_um2].copy()

    # restrict to chosen cell types for this dataset
    df = df[df['cell_type'].isin(allowed_types)].copy()

    # transcript density per cell
    df['transcript_density'] = df['nCount_RNA'] / df['nuc_area_um2']

    # enforce minimum cells per type
    vc = df['cell_type'].value_counts()
    keep_types = vc[vc >= min_cells].index.tolist()
    df = df[df['cell_type'].isin(keep_types)].copy()

    return df

df4 = build_cell_df(bgs4, 'GW24', cell_type_order_bgs4)
df5 = build_cell_df(bgs5, 'GW35', cell_type_order_bgs5)

# Summarize per cell type + compute residual vs size trend
def summarize_and_residualize(df):
    summ = (
        df.groupby('cell_type')
          .agg(
              nuc_area_med=('nuc_area_um2', 'median'),
              density_med=('transcript_density', 'median'),
              n_cells=('transcript_density', 'size')
          )
          .reset_index()
    )

    # Fit linear trend across cell types: density ~ area
    x = summ['nuc_area_med'].to_numpy()
    y = summ['density_med'].to_numpy()
    b, a = np.polyfit(x, y, 1)  # y ≈ b*x + a

    summ['density_fit'] = b * summ['nuc_area_med'] + a
    summ['density_resid'] = summ['density_med'] - summ['density_fit']

    return summ, (a, b)

sum4, (a4, b4) = summarize_and_residualize(df4)
sum5, (a5, b5) = summarize_and_residualize(df5)

# Plot: residual density ranking (labels moved further away)
def residual_barplot(summ, stage, colors, ax):
    summ2 = summ.sort_values('density_resid', ascending=True).copy()

    title_fs = 15
    label_fs = 12

    # symmetric x limits based on max residual magnitude
    m = np.nanmax(np.abs(summ2['density_resid'].to_numpy()))
    ax.set_xlim(-1.15 * m, 1.15 * m)

    # label offset based on x-range so it always looks good
    xspan = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = 0.008 * xspan  # <-- increase/decrease (0.03-0.07)

    ax.axvline(0, lw=2, alpha=0.5)

    for i, row in enumerate(summ2.itertuples(index=False)):
        ct = row.cell_type
        x = row.density_resid

        ax.scatter(
            x, i,
            s=70,
            color=colors.get(ct, 'gray'),
            edgecolor='black',
            linewidth=0.6,
            zorder=5
        )

        ax.text(
            x + (offset if x >= 0 else -offset),
            i,
            ct,
            ha='left' if x >= 0 else 'right',
            va='center',
            fontsize=label_fs,
            fontweight='normal'
        )

    ax.set_yticks([])
    ax.set_xlabel("Transcript density residual (median density − fitted size-trend)")
    ax.set_title(f"{stage}: density enrichment/depletion vs size trend",
                 fontsize=title_fs, fontweight='normal')
    ax.grid(axis='x', alpha=0.25, linestyle='--')

# Make figure
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'hspace': 0.35})

residual_barplot(sum4, "GW24 (bgs4)", cell_type_colors_bgs4, axes[0])
residual_barplot(sum5, "GW35 (bgs5)", cell_type_colors_bgs5, axes[1])

plt.tight_layout()
plt.show()

# Optional: print residual tables
print("\nGW24 residual density summary (positive = higher than expected for size):")
print(
    sum4[['cell_type', 'nuc_area_med', 'density_med', 'density_fit', 'density_resid', 'n_cells']]
    .sort_values('density_resid', ascending=False)
    .round(6)
    .to_string(index=False)
)

print("\nGW35 residual density summary (positive = higher than expected for size):")
print(
    sum5[['cell_type', 'nuc_area_med', 'density_med', 'density_fit', 'density_resid', 'n_cells']]
    .sort_values('density_resid', ascending=False)
    .round(6)
    .to_string(index=False)
)